<a href="https://colab.research.google.com/github/tanercc/python-colab/blob/dev/tjk_to_Predict.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q mysql-connector-python
!pip install pandas tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.0/34.0 MB 23.1 MB/s eta 0:00:00


Get TJK DB

In [2]:
import mysql.connector as connection
import pandas as pd
try:
    mydb = connection.connect(host="taner.web.tr", database = 'tanerweb_tjk',user="tanerweb_tjk", passwd="Ka9jVjRJRtRW",use_pure=True)
    query = """
SELECT
`atlar`.`KOD` AS `code`,
`atlar`.`ADKUCUK` AS `name`,
`kosular`.`GRUP_EN` AS `group`,
`atlar`.`ANNE` AS `mother`,
`atlar`.`BABA` AS `father`,
`kosular`.`TARIH` AS `date`,
`hipodrom`.`AD` AS `hipname`,
`hipodrom`.`KOD` AS `hipcode`,
`kosular`.`ActiveClass` AS `surface`,
COALESCE(CASE WHEN `kosular`.`PIST` = 'cim' THEN `hava`.`CIM_EN` ELSE `hava`.`KUM_EN` END, 'Normal') AS `ground`,
`kosular`.`RACENO` AS `raceno`,
`atlar`.`KILO` + `atlar`.`FAZLAKILO` AS `weight`,
`atlar`.`JOKEYADI` AS `jockey`,
`kosular`.`MESAFE` AS `metre`,
`atlar`.`DERECE` AS `time`,
COALESCE(`atlar`.`SONUC`, (SELECT COUNT(NO) FROM `atlar` AS `atsay` WHERE `atlar`.`TARIHKOD`=`atsay`.`TARIHKOD` AND `atlar`.`HIPODROMKOD`=`atsay`.`HIPODROMKOD` AND `atlar`.`KOSUNO`=`atsay`.`KOSUNO`)) AS `pos`
FROM `atlar`
LEFT JOIN `hava` ON (`atlar`.`TARIHKOD` = `hava`.`TARIHKOD` AND `atlar`.`HIPODROMKOD` = `hava`.`HIPODROMKOD`)
LEFT JOIN `kosular` ON (`kosular`.`TARIHKOD` = `atlar`.`TARIHKOD` AND `atlar`.`HIPODROMKOD` = `kosular`.`HIPODROMKOD` AND `atlar`.`KOSUNO` = `kosular`.`NO`)
LEFT JOIN `hipodrom` ON `hipodrom`.`KOD` = `atlar`.`HIPODROMKOD`
WHERE `atlar`.`TARIHKOD` > 20200904
AND `atlar`.`KILO` > 0
AND `atlar`.`KOSMAZ` = 0
AND `atlar`.`GECCIKIS_BOY` = ''
AND `atlar`.`START` > 0
AND `atlar`.`DERECE` > 0
ORDER BY `atlar`.`TARIHKOD` DESC,`atlar`.`HIPODROMKOD`,`atlar`.`KOSUNO`
    """
    #AND `atlar`.`HIPODROMKOD` = 1
    df = pd.read_sql(query,mydb)
    mydb.close() #close the connection
except Exception as e:
    mydb.close()
    print(str(e))

<ipython-input-2-5d1f8d96f011>:36: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query,mydb)


In [3]:
df.head()

,code,name,group,mother,father,date,hipname,hipcode,surface,ground,raceno,weight,jockey,metre,time,pos
0,102766,Adanalı Kız,3 Years Old Thoroughbreds,ADİLE TEYZE,BODEMEISTER (USA),29/03/2025,Adana Yeşiloba Hipodromu,1,sand,Good Going,1,60.0,AHMET ÇELİK,1400,93.55,8
1,102460,Beautiful Mina,3 Years Old Thoroughbreds,Mİ BOMBON,SUPER SAVER (USA),29/03/2025,Adana Yeşiloba Hipodromu,1,sand,Good Going,1,58.0,MEHMET KESKİN,1400,94.32,9
2,106207,Felix Perla,3 Years Old Thoroughbreds,COPY CAT,KANEKO,29/03/2025,Adana Yeşiloba Hipodromu,1,sand,Good Going,1,58.8,SELİM KAYA,1400,90.56,4
3,105513,Lady Artemis,3 Years Old Thoroughbreds,FLO JO,MENDIP (USA),29/03/2025,Adana Yeşiloba Hipodromu,1,sand,Good Going,1,56.8,HIŞMAN ÇİZİK,1400,91.46,6
4,104086,Mothers Love Mea,3 Years Old Thoroughbreds,LADY KENTUCKY (USA),GOOD CURRY,29/03/2025,Adana Yeşiloba Hipodromu,1,sand,Good Going,1,56.0,MÜSLÜM ÇELİK,1400,89.36,1


In [4]:
#df.to_csv("data-output.csv")

In [5]:
# Drop unnecessary columns (names, codes, date, etc.)
df = df.drop(columns=["code", "hipname", "hipcode", "date", "raceno", "pos"])

# Optional: Handle nulls
df = df.dropna()

df.head()

,name,group,mother,father,surface,ground,weight,jockey,metre,time
0,Adanalı Kız,3 Years Old Thoroughbreds,ADİLE TEYZE,BODEMEISTER (USA),sand,Good Going,60.0,AHMET ÇELİK,1400,93.55
1,Beautiful Mina,3 Years Old Thoroughbreds,Mİ BOMBON,SUPER SAVER (USA),sand,Good Going,58.0,MEHMET KESKİN,1400,94.32
2,Felix Perla,3 Years Old Thoroughbreds,COPY CAT,KANEKO,sand,Good Going,58.8,SELİM KAYA,1400,90.56
3,Lady Artemis,3 Years Old Thoroughbreds,FLO JO,MENDIP (USA),sand,Good Going,56.8,HIŞMAN ÇİZİK,1400,91.46
4,Mothers Love Mea,3 Years Old Thoroughbreds,LADY KENTUCKY (USA),GOOD CURRY,sand,Good Going,56.0,MÜSLÜM ÇELİK,1400,89.36


In [6]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.losses import MeanSquaredError

# Fix numerics
numeric_cols = ['weight', 'metre', 'time']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '').str.strip(), errors='coerce')
df = df.dropna(subset=numeric_cols)

# Categorical columns
cat_cols = ['name', 'group', 'mother', 'father', 'surface', 'ground', 'jockey']
num_cols = ['weight', 'metre']

# Features and label
X = df[cat_cols + num_cols]
y = df["time"]

# Preprocessing: OneHot for categoricals, Scale numerics
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ("num", StandardScaler(), num_cols)
])

X_processed = preprocessor.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)

# Build Neural Network model
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(64, activation='relu'),
    Dense(1)  # Output layer for regression
])

model.compile(
    optimizer=Adam(learning_rate=0.01),
    loss=MeanSquaredError(),
    metrics=['mae']
)

# Train the model
early_stop = EarlyStopping(patience=10, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=16,
    callbacks=[early_stop],
    verbose=1
)

# Evaluate
loss, mae = model.evaluate(X_test, y_test)
print(f"\n✅ Mean Absolute Error: {mae:.2f} seconds")


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
12179/12179 ━━━━━━━━━━━━━━━━━━━━ 47s 4ms/step - loss: 179.8906 - mae: 7.3132 - val_loss: 30.2485 - val_mae: 3.8871
Epoch 2/100
12179/12179 ━━━━━━━━━━━━━━━━━━━━ 43s 3ms/step - loss: 29.2595 - mae: 3.7475 - val_loss: 24.9587 - val_mae: 3.3365
Epoch 3/100
12179/12179 ━━━━━━━━━━━━━━━━━━━━ 42s 3ms/step - loss: 24.2777 - mae: 3.3129 - val_loss: 27.6928 - val_mae: 3.4312
Epoch 4/100
12179/12179 ━━━━━━━━━━━━━━━━━━━━ 85s 4ms/step - loss: 21.6259 - mae: 3.1032 - val_loss: 25.5678 - val_mae: 3.5216
Epoch 5/100
12179/12179 ━━━━━━━━━━━━━━━━━━━━ 45s 4ms/step - loss: 20.2568 - mae: 2.9373 - val_loss: 26.5202 - val_mae: 3.6572
Epoch 6/100
12179/12179 ━━━━━━━━━━━━━━━━━━━━ 40s 3ms/step - loss: 18.4811 - mae: 2.8217 - val_loss: 24.0440 - val_mae: 3.0937
Epoch 7/100
12179/12179 ━━━━━━━━━━━━━━━━━━━━ 40s 3ms/step - loss: 17.8012 - mae: 2.7104 - val_loss: 22.8329 - val_mae: 2.9991
Epoch 8/100
12179/12179 ━━━━━━━━━━━━━━━━━━━━ 42s 3ms/step - loss: 17.2606 - mae: 2.6438 - val_loss: 22.9870 - val_mae

In [8]:
# Save the model (full model: architecture + weights + optimizer)
model.save("race_time_model.h5")
print("✅ Model saved as race_time_model.h5")

✅ Model saved as race_time_model.h5


In [9]:
import joblib

# Save the preprocessor pipeline
joblib.dump(preprocessor, "preprocessor.pkl")
print("✅ Preprocessor saved as preprocessor.pkl")

✅ Preprocessor saved as preprocessor.pkl


In [10]:
# Create a new input DataFrame
new_data = pd.DataFrame([{
    "name": "Beautiful Mina",
    "group": "3 Years Old Thoroughbreds",
    "mother": "Mİ BOMBON",
    "father": "SUPER SAVER (USA)",
    "surface": "sand",
    "ground": "Good Going",
    "weight": 57.0,
    "jockey": "AHMET ÇELİK",
    "metre": 1400
}])

# Preprocess and predict
new_processed = preprocessor.transform(new_data)
predicted_time = model.predict(new_processed)
print("Predicted Time:", round(predicted_time[0][0], 2), "seconds")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
Predicted Time: 88.8 seconds


In [13]:
from tensorflow.keras.models import load_model
import joblib
import pandas as pd

# Load the model and preprocessor
model_new = load_model("race_time_model.h5")
preprocessor_new = joblib.load("preprocessor.pkl")

# Example: New input
new_data = pd.DataFrame([{
    "name": "Beautiful Mina",
    "group": "3 Years Old Thoroughbreds",
    "mother": "Mİ BOMBON",
    "father": "SUPER SAVER (USA)",
    "surface": "sand",
    "ground": "Good Going",
    "weight": 57.0,
    "jockey": "AHMET ÇELİK",
    "metre": 1400
}])

# Transform and predict
new_processed = preprocessor_new.transform(new_data)
predicted_time = model_new.predict(new_processed)

print("🏁 Predicted Time:", round(predicted_time[0][0], 2), "seconds")

TypeError: Could not locate function 'mse'. Make sure custom classes are decorated with `@keras.saving.register_keras_serializable()`. Full object config: {'module': 'keras.metrics', 'class_name': 'function', 'config': 'mse', 'registered_name': 'mse'}